# PadelVision — 00 Setup check (Kaggle / Colab)

Verifies that a fresh notebook can install the pipeline, download models, read your video and run the
person detector on the GPU. It also saves one frame as `court_frame.png`; you'll click the court corners on it in phase 1.

**Kaggle:** Settings → Accelerator **GPU T4 x2**, **Internet on**. Add your match video as a Dataset (Add Input).
**Colab:** Runtime → Change runtime type → **T4 GPU**. Put your video in Google Drive.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

Set the path to your match video:

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Kaggle example: "/kaggle/input/<your-dataset>/match1.mp4"
# Colab example:  "/content/drive/MyDrive/padel/match1.mp4"
VIDEO_PATH = ""
assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "Set VIDEO_PATH to an existing video file"

In [ ]:
from padelvision.video import probe
info = probe(VIDEO_PATH)
info.to_dict()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from padelvision.models import weights_path

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, info.frame_count // 2)
ok, frame = cap.read()
cap.release()
assert ok, "Could not read a frame"

cv2.imwrite(f"{WORK}/court_frame.png", frame)

model = YOLO(str(weights_path("person-detector")))
result = model.predict(frame, classes=[0], conf=0.3, verbose=False)[0]
print(f"device={model.device}  people detected={len(result.boxes)}")

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off");

**Expected:** a GPU device (`cuda:0`), the video's real fps and resolution, and boxes on the 4 players
(plus maybe spectators; phase 1 filters those out using the court). Download `court_frame.png` from the output panel.